In [ ]:
# 1. Verifikasi Lingkungan GPU & Pustaka Pendukung
!nvidia-smi
!pip install -q timm thop scikit-learn pandas matplotlib seaborn


# Pelatihan & Benchmark Komparasi Arsitektur CASCADE Two-Stage (16 Kelas)

Dokumen notebook ini memodifikasi pipeline resmi **CASCADE Two-Stage (Stage 1 Jenis & Stage 2 Sangrai)** untuk melatih dan mengevaluasi secara langsung (*real training, fair comparison*) seluruh arsitektur backbone dari awal tanpa injeksi nilai tiruan:

### 1. Daftar 7 Arsitektur Backbone yang Dilatih:
1. **`convnext_tiny`**: ConvNeXt-Tiny (Baseline Proyek — Vision Transformer-inspired pure CNN).
2. **`resnet50`**: ResNet-50 (Residual connection standar industri, 50 layers).
3. **`resnet101`**: ResNet-101 (Arsitektur residual lebih dalam, 101 layers).
4. **`efficientnet_b0`**: EfficientNet-B0 (Compound scaling depth/width/resolution).
5. **`efficientnetv2_rw_m`**: EfficientNetV2-M (Fused-MBConv architecture).
6. **`mobilenetv3_large_100`**: MobileNetV3-Large (Optimasi mobile/edge dengan hard-swish & SE blocks).
7. **`densenet121`**: DenseNet-121 (Dense feature reuse antar seluruh lapisan).

### 2. Standarisasi Eksperimen (Fair Comparison):
- **Dataset & Pembagian Data**: Stratified 5-Fold Cross Validation identik (`Dataset_Split2`), 1.600 citra (1.280 train, 320 val per fold).
- **Resolusi & Normalisasi**: $224 \times 224$ piksel, ImageNet mean `[0.485, 0.456, 0.406]` & std `[0.229, 0.224, 0.225]`.
- **Augmentasi Terukur**: Rotasi $15^\circ$, ColorJitter ($0.05, 0.05$), Random Horizontal & Vertical Flip ($0.5$).
- **Classifier Head Terstandarisasi**: $\text{Linear}(in\_features, 512) \rightarrow \text{LayerNorm}(512) \rightarrow \text{GELU}() \rightarrow \text{Dropout}(0.3) \rightarrow \text{Linear}(512, 4)$.
- **Optimizer & LR**: AdamW dengan 2-phase warmup 8 epoch (freeze backbone), dilanjutkan unfreeze dengan differential learning rate (`backbone: 3e-5`, `head: 2.5e-4`), CosineAnnealingLR, dan label smoothing 0.1.
- **Reproducibility**: Random Seed ditetapkan permanen pada 42.
- **Penyimpanan Terpisah**: Bobot model (`.pth`), grafik riwayat epoch (`.png`), confusion matrix (`.png` 300 DPI & `.csv`), dan classification report (`.csv`) disimpan dalam direktori terisolasi untuk masing-masing backbone.


## 1. Import Libraries, Random Seed & Pengaturan Direktori Lingkungan


In [ ]:
import os
import sys
import json
import time
import shutil
import random
import zipfile
import urllib.request
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
from torch.optim.lr_scheduler import CosineAnnealingLR

import timm
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False

# --- REPRODUCIBILITY SEED ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# --- DEVICE ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[INFO] Running on Device: {device}')
if torch.cuda.is_available():
    print(f'[INFO] GPU: {torch.cuda.get_device_name(0)}')

# --- ENVIRONMENT & WORKING DIRECTORIES ---
if os.path.exists('/kaggle/working'):
    WORK_DIR = '/kaggle/working'
    INPUT_BASE = '/kaggle/input'
elif os.path.exists('/content'):
    WORK_DIR = '/content'
    INPUT_BASE = '/content'
else:
    WORK_DIR = os.getcwd()
    INPUT_BASE = os.getcwd()

# Cari dataset secara dinamis atau unduh otomatis
HF_DATASET_URL = 'https://huggingface.co/turasa22/klasifikasibijikopiconvnext/resolve/main/Dataset_Split2.zip'
TARGET_SPLIT2_DIR = None

candidate_paths = [
    os.path.join(WORK_DIR, 'Dataset_Split2'),
    os.path.join(INPUT_BASE, 'Dataset_Split2'),
    'D:/Riset_Mbkm_2026/PROJECT_1V/final50epoch/Dataset_Split2',
    '../PROJECT_1V/final50epoch/Dataset_Split2',
]
for p in candidate_paths:
    if os.path.isdir(p) and os.path.exists(os.path.join(p, 'fold_1')):
        TARGET_SPLIT2_DIR = os.path.abspath(p)
        break

if TARGET_SPLIT2_DIR is None:
    for root, dirs, files in os.walk(INPUT_BASE):
        if 'fold_1' in dirs and 'fold_2' in dirs:
            TARGET_SPLIT2_DIR = root
            break

if TARGET_SPLIT2_DIR is None:
    zip_dest = os.path.join(WORK_DIR, 'Dataset_Split2.zip')
    print(f'[INFO] Mengunduh dataset resmi dari Hugging Face ({HF_DATASET_URL})...')
    urllib.request.urlretrieve(HF_DATASET_URL, zip_dest)
    with zipfile.ZipFile(zip_dest, 'r') as zf:
        zf.extractall(WORK_DIR)
    for root, dirs, files in os.walk(WORK_DIR):
        if 'fold_1' in dirs and 'fold_2' in dirs:
            TARGET_SPLIT2_DIR = root
            break

MODELS_BASE_DIR = os.path.join(WORK_DIR, 'models', 'CASCADE')
REPORTS_BASE_DIR = os.path.join(WORK_DIR, 'reports', 'CASCADE')
os.makedirs(MODELS_BASE_DIR, exist_ok=True)
os.makedirs(REPORTS_BASE_DIR, exist_ok=True)

print(f'[INFO] Direktori Dataset: {TARGET_SPLIT2_DIR}')
print(f'[INFO] Simpan Model Ke  : {MODELS_BASE_DIR}')
print(f'[INFO] Simpan Report Ke : {REPORTS_BASE_DIR}')


## 2. Pemetaan Label & Verifikasi Partisi Fold


In [ ]:
# --- DEFINISI LABEL RESMI (IDENTIK DENGAN PROTOKOL ASLI) ---
JENIS_LABELS   = {'arabika': 0, 'ekselsa': 1, 'liberika': 2, 'robusta': 3}
SANGRAI_LABELS = {'dark': 0, 'green': 1, 'light': 2, 'medium': 3}

SPECIES_NAMES = ['Arabika', 'Ekselsa', 'Liberika', 'Robusta']
ROAST_NAMES   = ['Dark', 'Green', 'Light', 'Medium']

CLASS_NAMES_16 = [
    f'{j}_{s}'
    for j in SPECIES_NAMES
    for s in ROAST_NAMES
]

print(f'[INFO] Status Ketersediaan Fold di: {TARGET_SPLIT2_DIR}')
if TARGET_SPLIT2_DIR:
    for fold_idx in range(1, 6):
        f_path = os.path.join(TARGET_SPLIT2_DIR, f'fold_{fold_idx}')
        tr_cnt = sum(len(files) for _, _, files in os.walk(os.path.join(f_path, 'train'))) if os.path.exists(os.path.join(f_path, 'train')) else 0
        va_cnt = sum(len(files) for _, _, files in os.walk(os.path.join(f_path, 'val'))) if os.path.exists(os.path.join(f_path, 'val')) else 0
        print(f'  Fold {fold_idx}: Train = {tr_cnt} citra | Val = {va_cnt} citra (Total = {tr_cnt + va_cnt})')


## 3. Augmentasi Citra & Arsitektur Backbone Modular

### Penjelasan Perubahan & Alasan Teknis:
1. **Perbedaan Dimensi Fitur (`in_features`)**: Setiap arsitektur memiliki representasi dimensi laten akhir yang berbeda setelah Global Average Pooling:
   - `convnext_tiny`: 768 dimensi
   - `resnet50` & `resnet101`: 2048 dimensi
   - `efficientnet_b0`: 1280 dimensi
   - `efficientnetv2_rw_m`: 2152 dimensi
   - `mobilenetv3_large_100`: 1280 dimensi
   - `densenet121`: 1024 dimensi
2. **Standarisasi Classifier Head**: Untuk memastikan perbandingan yang adil, seluruh arsitektur (termasuk ConvNeXt-Tiny) menggunakan classifier head dengan struktur identik:
   $$\text{Linear}(in\_features, 512) \rightarrow \text{LayerNorm}(512) \rightarrow \text{GELU}() \rightarrow \text{Dropout}(0.3) \rightarrow \text{Linear}(512, 4)$$
   Setiap backbone dihilangkan classifier bawaannya (`reset_classifier(0)`) lalu disambungkan ke head terstandarisasi ini.


In [ ]:
def get_transforms(img_size=224):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.05, contrast=0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return train_tf, val_tf

# --- REGISTRI ARSITEKTUR BACKBONE ---
BACKBONE_REGISTRY = {
    'convnext_tiny': {
        'name': 'ConvNeXt-Tiny (Baseline)',
        'family': 'ConvNeXt',
        'timm_name': 'convnext_tiny',
        'desc': 'Modern pure CNN baseline incorporating Vision Transformer design.'
    },
    'resnet50': {
        'name': 'ResNet-50',
        'family': 'ResNet',
        'timm_name': 'resnet50',
        'desc': 'Residual learning 50 layers with bottleneck skip-connections.'
    },
    'resnet101': {
        'name': 'ResNet-101',
        'family': 'ResNet',
        'timm_name': 'resnet101',
        'desc': 'Deeper residual network (101 layers) for higher feature capacity.'
    },
    'efficientnet_b0': {
        'name': 'EfficientNet-B0',
        'family': 'EfficientNet V1',
        'timm_name': 'efficientnet_b0',
        'desc': 'Compound scaling depth, width, and resolution (lightweight).'
    },
    'efficientnetv2_rw_m': {
        'name': 'EfficientNetV2-M',
        'family': 'EfficientNet V2',
        'timm_name': 'efficientnetv2_rw_m',
        'desc': 'Fused-MBConv architecture with improved training speed and accuracy.'
    },
    'mobilenetv3_large_100': {
        'name': 'MobileNetV3-Large',
        'family': 'MobileNet',
        'timm_name': 'mobilenetv3_large_100',
        'desc': 'Mobile/Edge optimized CNN with hard-swish and squeeze-and-excitation.'
    },
    'densenet121': {
        'name': 'DenseNet-121',
        'family': 'DenseNet',
        'timm_name': 'densenet121',
        'desc': 'Dense feature connectivity with direct feed-forward reuse.'
    }
}

class UniversalCascadeClassifier(nn.Module):
    def __init__(self, backbone_key='convnext_tiny', num_classes=4, drop_rate=0.3, pretrained=True):
        super().__init__()
        self.backbone_key = backbone_key
        cfg = BACKBONE_REGISTRY[backbone_key]
        self.backbone = timm.create_model(cfg['timm_name'], pretrained=pretrained)
        in_features = self.backbone.get_classifier().in_features
        self.backbone.reset_classifier(0)  # Hapus classification head bawaan
        
        # Standarisasi classifier head sesuai protokol resmi
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(drop_rate),
            nn.Linear(512, num_classes)
        )
        self.in_features = in_features

    def forward(self, x):
        feat = self.backbone(x)
        return self.head(feat)

print('Verifikasi Instansiasi Head 4-Kelas CASCADE untuk Setiap Arsitektur:')
for k, v in BACKBONE_REGISTRY.items():
    m = UniversalCascadeClassifier(backbone_key=k, num_classes=4, pretrained=False)
    print(f"  [{k:22s}] in_features: {m.in_features:4d} -> Head: 512 -> 4 Kelas | {v['name']}")


## 4. Engine Pelatihan (2-Phase Warmup, Differential Learning Rate & Cosine Annealing)

### Penjelasan Teknis Adaptasi Optimizer:
Pada kode ConvNeXt asli, layer stem dan stages diakses dengan nama atribut spesifik (`model.stem`, `model.stages[0..3]`). Agar berlaku universal untuk seluruh arsitektur, parameter dibagi ke dalam 2 kelompok bersih:
- **Fase 1 (Warmup: Epoch 0 sampai 7)**: Seluruh parameter backbone dibekukan (`requires_grad = False`). Hanya classifier head yang dilatih dengan $\text{lr} = 5 \times 10^{-4}$ dan $\text{weight\_decay} = 10^{-2}$.
- **Fase 2 (Unfreeze: Epoch 8 ke atas)**: Seluruh parameter dibuka dengan *differential learning rate*:
  - $\text{Backbone Parameter Group}$: $\text{lr} = 3 \times 10^{-5}$ (menjaga bobot representasi pretrained ImageNet agar tidak mengalami *catastrophic forgetting*).
  - $\text{Head Parameter Group}$: $\text{lr} = 2.5 \times 10^{-4}$.
  - $\text{Scheduler}$: Cosine Annealing LR dengan $\eta_{\min} = 10^{-6}$.


In [ ]:
def train_kfold_model(
    backbone_key,
    train_loader,
    val_loader,
    num_classes=4,
    save_name='model.pth',
    label_info='Fold 1',
    target_name=None,
    history_save_dir=None,
    num_epochs=100,
    warmup_epochs=8,
    patience=15
):
    model = UniversalCascadeClassifier(backbone_key=backbone_key, num_classes=num_classes, pretrained=True).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    display_target = target_name if target_name is not None else label_info

    # --- FASE 1: WARMUP (Freeze Backbone) ---
    for p in model.backbone.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True

    optimizer = optim.AdamW(
        model.head.parameters(),
        lr=5e-4, weight_decay=1e-2
    )
    scheduler = None

    best_acc = 0.0
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    print(f'\n[INFO] Device: {device} | Backbone: {BACKBONE_REGISTRY[backbone_key]["name"]} | Target: {display_target}')
    print(f'{"="*70}')
    print(f'  MULAI TRAINING: {label_info} | Output: {num_classes} Kelas | Max Epoch: {num_epochs}')
    print(f'{"="*70}')

    for epoch in range(num_epochs):
        # --- FASE 2: UNFREEZE (Differential Learning Rate) ---
        if epoch == warmup_epochs:
            for p in model.backbone.parameters():
                p.requires_grad = True

            optimizer = optim.AdamW([
                {'params': model.backbone.parameters(), 'lr': 3.0e-5},
                {'params': model.head.parameters(),     'lr': 2.5e-4},
            ], weight_decay=1e-2)

            scheduler = CosineAnnealingLR(optimizer, T_max=(num_epochs - warmup_epochs), eta_min=1e-6)

        # Training Phase
        model.train()
        running_loss, train_correct, train_total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss  += loss.item() * inputs.size(0)
            _, preds       = torch.max(outputs, 1)
            train_total   += labels.size(0)
            train_correct += (preds == labels).sum().item()

        if epoch >= warmup_epochs and scheduler is not None:
            scheduler.step()

        epoch_tr_loss = running_loss / len(train_loader.dataset)
        epoch_tr_acc  = train_correct / train_total

        # Validation Phase
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss    = criterion(outputs, labels)

                val_loss    += loss.item() * inputs.size(0)
                _, preds     = torch.max(outputs, 1)
                val_total   += labels.size(0)
                val_correct += (preds == labels).sum().item()

        epoch_va_loss = val_loss / len(val_loader.dataset)
        epoch_va_acc  = val_correct / val_total

        history['train_loss'].append(epoch_tr_loss)
        history['train_acc'].append(epoch_tr_acc)
        history['val_loss'].append(epoch_va_loss)
        history['val_acc'].append(epoch_va_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == num_epochs - 1:
            print(f"{label_info} | Epoch {epoch+1:02d}/{num_epochs:02d} | Train Loss: {epoch_tr_loss:.4f} Acc: {epoch_tr_acc*100:.2f}% | Val Loss: {epoch_va_loss:.4f} Acc: {epoch_va_acc*100:.2f}%")

        if epoch_va_acc > best_acc:
            best_acc = epoch_va_acc
            torch.save(model.state_dict(), save_name)

        if epoch_va_loss < best_val_loss:
            best_val_loss = epoch_va_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience and epoch >= warmup_epochs + 7:
                print(f"[Early Stopping] {label_info} berhenti pada epoch {epoch+1}. Best Val Acc: {best_acc*100:.2f}%")
                break

    print(f"=> {label_info} Selesai! Best Val Accuracy: {best_acc*100:.2f}%")

    # Simpan Grafik Loss & Accuracy jika direktori disediakan
    if history_save_dir is not None:
        os.makedirs(history_save_dir, exist_ok=True)
        epochs_range = range(1, len(history['train_loss']) + 1)
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(epochs_range, history['train_loss'], label='Train Loss')
        plt.plot(epochs_range, history['val_loss'],   label='Val Loss')
        plt.legend()
        plt.title(f"{BACKBONE_REGISTRY[backbone_key]['name']} — Loss ({display_target})")
        plt.grid(True)

        plt.subplot(1, 2, 2)
        plt.plot(epochs_range, history['train_acc'], label='Train Acc')
        plt.plot(epochs_range, history['val_acc'],   label='Val Acc')
        plt.legend()
        plt.title(f"{BACKBONE_REGISTRY[backbone_key]['name']} — Accuracy ({display_target})")
        plt.grid(True)
        plt.tight_layout()

        safe_name = label_info.replace(' ', '_').replace('/', '_')
        plot_path = os.path.join(history_save_dir, f"History_{safe_name}.png")
        try:
            plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        except Exception:
            pass
        plt.close()

    return history, best_acc


## 5. Eksekusi Utama: CASCADE 5-Fold Cross Validation untuk Seluruh 7 Backbone

Seluruh arsitektur (termasuk baseline **ConvNeXt-Tiny**) dilatih dari awal secara adil dan nyata tanpa injeksi nilai:
1. `convnext_tiny` (ConvNeXt-Tiny — Baseline)
2. `resnet50` (ResNet-50)
3. `resnet101` (ResNet-101)
4. `efficientnet_b0` (EfficientNet V1)
5. `efficientnetv2_rw_m` (EfficientNet V2)
6. `mobilenetv3_large_100` (MobileNetV3)
7. `densenet121` (DenseNet-121)

> Anda dapat mengatur `RUN_BACKBONES` dan `FOLDS_TO_RUN` sesuai kebutuhan alokasi waktu dan kuota GPU Anda.


In [ ]:
# --- PENGATURAN EKSEKUSI EXPERIMEN CASCADE LENGKAP ---
# Seluruh 7 backbone dilatih secara nyata dari awal (tanpa injeksi):
RUN_BACKBONES = [
    'convnext_tiny',        # ConvNeXt-Tiny (Baseline)
    'resnet50',             # ResNet-50
    'resnet101',            # ResNet-101
    'efficientnet_b0',      # EfficientNet V1
    'efficientnetv2_rw_m',  # EfficientNet V2
    'mobilenetv3_large_100',# MobileNetV3
    'densenet121'           # DenseNet-121
]

FOLDS_TO_RUN = range(1, 6)   # 5-Fold Lengkap (1 sampai 5)
NUM_EPOCHS   = 100          # 100 Epochs per submodel (sesuai protokol resmi)
WARMUP_EPOCHS = 8           # 8 Epochs Warmup (freeze backbone)
PATIENCE     = 15           # Patience early stopping
BATCH_SIZE   = 32

train_tf, val_tf = get_transforms(img_size=224)
global_backbone_summary = []

for b_key in RUN_BACKBONES:
    b_info = BACKBONE_REGISTRY[b_key]
    print(f'\n{"="*85}')
    print(f'       MEMULAI CASCADE 5-FOLD: {b_info["name"]} ({b_info["family"]})')
    print(f'{"="*85}')
    
    b_models_dir = os.path.join(MODELS_BASE_DIR, b_key)
    b_reports_dir = os.path.join(REPORTS_BASE_DIR, b_key)
    b_hist_dir = os.path.join(b_reports_dir, 'Training_Histories')
    os.makedirs(b_models_dir, exist_ok=True)
    os.makedirs(b_reports_dir, exist_ok=True)
    os.makedirs(b_hist_dir, exist_ok=True)
    
    cascade_kfold_metrics = []
    oof_casc_targets = []
    oof_casc_preds = []
    stage1_summary = []
    t_backbone_start = time.time()
    
    for fold_num in FOLDS_TO_RUN:
        fold_train_root = os.path.join(TARGET_SPLIT2_DIR, f'fold_{fold_num}', 'train')
        fold_val_root   = os.path.join(TARGET_SPLIT2_DIR, f'fold_{fold_num}', 'val')
        
        print(f'\n[CASCADE {b_info["name"]}] --- FOLD {fold_num} ---')
        
        # ==========================================================================
        # [STAGE 1] TRAINING MODEL JENIS (4 KELAS SPESIES)
        # ==========================================================================
        print(f'  [Stage 1] Training Model Jenis (4 Spesies)...')
        ds_j_train = datasets.ImageFolder(fold_train_root, transform=train_tf)
        ds_j_val   = datasets.ImageFolder(fold_val_root,   transform=val_tf)
        
        loader_j_tr = DataLoader(ds_j_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
        loader_j_va = DataLoader(ds_j_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
        
        m_j_path = os.path.join(b_models_dir, f'{b_key}_Jenis_Fold{fold_num}.pth')
        history_j, jenis_best_acc = train_kfold_model(
            backbone_key=b_key,
            train_loader=loader_j_tr,
            val_loader=loader_j_va,
            num_classes=4,
            save_name=m_j_path,
            label_info=f'{b_key} Jenis Fold {fold_num}',
            target_name='Jenis',
            history_save_dir=b_hist_dir,
            num_epochs=NUM_EPOCHS,
            warmup_epochs=WARMUP_EPOCHS,
            patience=PATIENCE
        )
        stage1_summary.append({
            'Fold': f'Fold {fold_num}',
            'Jenis Accuracy (%)': round(jenis_best_acc * 100, 2)
        })
        
        # ==========================================================================
        # [STAGE 2] TRAINING 4 MODEL SANGRAI PER JENIS (4 KELAS x 4 SPESIES)
        # ==========================================================================
        print(f'  [Stage 2] Training 4 Model Sangrai...')
        m_roast_paths = {}
        for j_name in SPECIES_NAMES:
            j_tr_path = os.path.join(fold_train_root, j_name)
            j_va_path = os.path.join(fold_val_root,   j_name)
            
            ds_r_tr = datasets.ImageFolder(j_tr_path, transform=train_tf)
            ds_r_va = datasets.ImageFolder(j_va_path, transform=val_tf)
            
            loader_r_tr = DataLoader(ds_r_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
            loader_r_va = DataLoader(ds_r_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
            
            m_r_path = os.path.join(b_models_dir, f'{b_key}_{j_name}_Fold{fold_num}.pth')
            m_roast_paths[j_name] = m_r_path
            
            train_kfold_model(
                backbone_key=b_key,
                train_loader=loader_r_tr,
                val_loader=loader_r_va,
                num_classes=4,
                save_name=m_r_path,
                label_info=f'{b_key} {j_name} Fold {fold_num}',
                target_name=j_name,
                history_save_dir=b_hist_dir,
                num_epochs=NUM_EPOCHS,
                warmup_epochs=WARMUP_EPOCHS,
                patience=PATIENCE
            )
            
        # ==========================================================================
        # [EVALUASI BERANTAI CASCADE 16 KELAS]
        # Input -> Prediksi Jenis -> Routing ke Model Sangrai -> Gabung 16 Kelas
        # ==========================================================================
        print(f'  [Evaluasi Berantai 16 Kelas Fold {fold_num}]...')
        m_j = UniversalCascadeClassifier(backbone_key=b_key, num_classes=4, pretrained=False).to(device)
        m_j.load_state_dict(torch.load(m_j_path, map_location=device))
        m_j.eval()
        
        m_r = {}
        for j_n in SPECIES_NAMES:
            m = UniversalCascadeClassifier(backbone_key=b_key, num_classes=4, pretrained=False).to(device)
            m.load_state_dict(torch.load(m_roast_paths[j_n], map_location=device))
            m_r[j_n] = m.eval()
            
        y_t_casc, y_p_casc = [], []
        for j_idx, j_n in enumerate(SPECIES_NAMES):
            for s_idx, s_n in enumerate(ROAST_NAMES):
                sub_val_dir = os.path.join(fold_val_root, j_n, s_n)
                if not os.path.exists(sub_val_dir):
                    continue
                expected_flat = (j_idx * 4) + s_idx
                for img_f in sorted(os.listdir(sub_val_dir)):
                    if not img_f.lower().endswith(('.jpg', '.jpeg', '.png')):
                        continue
                    img = Image.open(os.path.join(sub_val_dir, img_f)).convert('RGB')
                    tensor = val_tf(img).unsqueeze(0).to(device)
                    with torch.no_grad():
                        pred_j_idx = torch.argmax(m_j(tensor), 1).item()
                        pred_j_name = SPECIES_NAMES[pred_j_idx]
                        pred_s_idx = torch.argmax(m_r[pred_j_name](tensor), 1).item()
                        pred_flat = (pred_j_idx * 4) + pred_s_idx
                    
                    y_t_casc.append(expected_flat)
                    y_p_casc.append(pred_flat)
                    
        c_acc  = accuracy_score(y_t_casc, y_p_casc) * 100
        c_prec = precision_score(y_t_casc, y_p_casc, average='macro', zero_division=0) * 100
        c_rec  = recall_score(y_t_casc, y_p_casc, average='macro', zero_division=0) * 100
        c_f1_m = f1_score(y_t_casc, y_p_casc, average='macro', zero_division=0) * 100
        c_f1_w = f1_score(y_t_casc, y_p_casc, average='weighted', zero_division=0) * 100
        
        print(f'  [Fold {fold_num} Selesai] Jenis Acc: {jenis_best_acc*100:.2f}% | 16-Cls Acc: {c_acc:.2f}% | F1 Macro: {c_f1_m:.2f}%')
        oof_casc_targets.extend(y_t_casc)
        oof_casc_preds.extend(y_p_casc)
        cascade_kfold_metrics.append({
            'Fold': f'Fold {fold_num}',
            'Stage 1 Jenis Acc (%)': round(jenis_best_acc * 100, 2),
            'Accuracy (%)': round(c_acc, 2),
            'Precision Macro (%)': round(c_prec, 2),
            'Recall Macro (%)': round(c_rec, 2),
            'F1 Macro (%)': round(c_f1_m, 2),
            'F1 Weighted (%)': round(c_f1_w, 2)
        })
        
    t_backbone_total = time.time() - t_backbone_start
    
    # Simpan Ringkasan Stage 1
    df_stage1 = pd.DataFrame(stage1_summary)
    df_stage1.to_csv(os.path.join(b_reports_dir, f'Cascade_Jenis_Summary_{b_key}.csv'), index=False)
    
    # Tabel Rekapitulasi 5-Fold Backbone Ini
    df_casc_indiv = pd.DataFrame(cascade_kfold_metrics)
    mean_r = {'Fold': 'MEAN (μ)'}
    std_r  = {'Fold': 'STD (σ)'}
    for col in [c for c in df_casc_indiv.columns if c != 'Fold']:
        mean_r[col] = round(df_casc_indiv[col].mean(), 2)
        std_r[col]  = round(df_casc_indiv[col].std(), 2)
    df_casc_indiv_final = pd.concat([df_casc_indiv, pd.DataFrame([mean_r, std_r])], ignore_index=True)
    df_casc_indiv_final.to_csv(os.path.join(b_reports_dir, f'Cascade_KFold_Metrics_Summary_{b_key}.csv'), index=False)
    
    # Confusion Matrix OOF (16x16) & Classification Report OOF
    cm_oof = confusion_matrix(oof_casc_targets, oof_casc_preds)
    df_cm = pd.DataFrame(cm_oof, index=CLASS_NAMES_16, columns=CLASS_NAMES_16)
    df_cm.index.name = 'True_Class'
    df_cm.to_csv(os.path.join(b_reports_dir, f'KFold_ConfusionMatrix_OOF_CASCADE_{b_key}.csv'))
    
    # Plot PNG Confusion Matrix 300 DPI
    fig, ax = plt.subplots(figsize=(14, 12))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_oof, display_labels=CLASS_NAMES_16)
    disp.plot(cmap='Blues', ax=ax, xticks_rotation=45, values_format='d', colorbar=True)
    plt.title(f'Out-of-Fold Confusion Matrix — {b_info["name"]} CASCADE (16 Kelas)', fontsize=13, pad=15)
    plt.tight_layout()
    plt.savefig(os.path.join(b_reports_dir, f'KFold_ConfusionMatrix_OOF_CASCADE_{b_key}.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # Classification Report CSV
    rep_dict = classification_report(oof_casc_targets, oof_casc_preds, target_names=CLASS_NAMES_16, output_dict=True)
    df_rep = pd.DataFrame(rep_dict).T
    df_rep.index.name = 'Class'
    df_rep.to_csv(os.path.join(b_reports_dir, f'KFold_ClassificationReport_OOF_CASCADE_{b_key}.csv'), float_format='%.4f')
    
    # Hitung Kompleksitas dan Latensi 2-Stage
    dummy = torch.randn((1, 3, 224, 224), device=device)
    m_dummy = UniversalCascadeClassifier(backbone_key=b_key, num_classes=4, pretrained=False).to(device)
    m_dummy.eval()
    single_p = sum(p.numel() for p in m_dummy.parameters()) / 1e6
    total_casc_p = single_p * 5
    
    gflops_2stage = 0.0
    if HAS_THOP:
        try:
            macs, _ = thop_profile(m_dummy, inputs=(dummy,), verbose=False)
            gflops_2stage = ((macs * 2) * 2) / 1e9
        except Exception:
            pass
            
    with torch.no_grad():
        for _ in range(5): _ = m_dummy(dummy)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0_lat = time.perf_counter()
        for _ in range(30):
            _ = m_dummy(dummy)
            _ = m_dummy(dummy)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t1_lat = time.perf_counter()
    lat_ms = ((t1_lat - t0_lat) / 30) * 1000
    fps = 1000.0 / lat_ms if lat_ms > 0 else 0.0
    
    global_backbone_summary.append({
        'Backbone': b_key,
        'Model_Name': b_info['name'],
        'Family': b_info['family'],
        'Stage1_Species_Acc (%)': mean_r['Stage 1 Jenis Acc (%)'],
        'Chained_16Cls_Acc (%)': mean_r['Accuracy (%)'],
        'Chained_F1_Macro (%)': mean_r['F1 Macro (%)'],
        'Chained_Precision_Macro (%)': mean_r['Precision Macro (%)'],
        'Chained_Recall_Macro (%)': mean_r['Recall Macro (%)'],
        'Chained_F1_Weighted (%)': mean_r['F1 Weighted (%)'],
        'Params_Single_M': round(single_p, 2),
        'Cascade_Total_Params_M': round(total_casc_p, 2),
        '2Stage_GFLOPs': round(gflops_2stage, 2),
        '2Stage_Latency_ms': round(lat_ms, 2),
        'Cascade_FPS': round(fps, 1),
        'Total_Train_Time_s': round(t_backbone_total, 1)
    })
    
    print(f'\n[SELESAI {b_info["name"]}] Mean 16-Cls Acc: {mean_r["Accuracy (%)"]}% | Mean F1 Macro: {mean_r["F1 Macro (%)"]}%')


## 6. Tabel Perbandingan Seluruh Backbone terhadap Baseline ConvNeXt-Tiny

Hasil seluruh model yang telah dilatih nyata disusun ke dalam tabel komparasi. Model `ConvNeXt-Tiny` secara otomatis dijadikan basis komparasi (Baseline) untuk menghitung selisih kinerja ($\Delta \text{Akurasi}$ dan $\Delta \text{F1 Macro}$).


In [ ]:
# --- TABEL SINTESIS KOMPARASI LINTAS BACKBONE (HASIL PELATIHAN AKTUAL) ---
df_all = pd.DataFrame(global_backbone_summary)

# Hitung Delta terhadap ConvNeXt-Tiny Baseline hasil training aktual
if 'convnext_tiny' in df_all['Backbone'].values:
    base_row = df_all[df_all['Backbone'] == 'convnext_tiny'].iloc[0]
    base_acc = base_row['Chained_16Cls_Acc (%)']
    base_f1  = base_row['Chained_F1_Macro (%)']
    df_all['Delta_Acc_vs_Baseline (%)'] = df_all['Chained_16Cls_Acc (%)'].apply(lambda x: round(x - base_acc, 2))
    df_all['Delta_F1_vs_Baseline (%)']  = df_all['Chained_F1_Macro (%)'].apply(lambda x: round(x - base_f1, 2))

df_all = df_all.sort_values(by='Chained_F1_Macro (%)', ascending=False).reset_index(drop=True)
df_all.index += 1
df_all.index.name = 'Rank'

print('\n' + '='*105)
print('       TABEL KOMPARASI ARSITEKTUR BACKBONE DALAM CASCADE CLASSIFIER (16 KELAS)')
print('='*105)
show_cols = [
    'Model_Name', 'Family', 'Stage1_Species_Acc (%)', 'Chained_16Cls_Acc (%)',
    'Chained_F1_Macro (%)', '2Stage_GFLOPs', '2Stage_Latency_ms', 'Cascade_FPS'
]
if 'Delta_Acc_vs_Baseline (%)' in df_all.columns:
    show_cols.extend(['Delta_Acc_vs_Baseline (%)', 'Delta_F1_vs_Baseline (%)'])
print(df_all[show_cols].to_string())
print('='*105)

summary_csv = os.path.join(REPORTS_BASE_DIR, 'CASCADE_All_Backbones_Comparison.csv')
df_all.to_csv(summary_csv)
print(f'\n[SAVED] Rekapitulasi CSV disimpan di: {summary_csv}')


## 7. Visualisasi Hasil Komparasi & Trade-off Efisiensi


In [ ]:
# --- VISUALISASI PERBANDINGAN DAN FRONTIER EFISIENSI ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Bar Chart: Akurasi & F1 Macro per Backbone
x_pos = np.arange(len(df_all))
width = 0.35

axes[0].bar(x_pos - width/2, df_all['Chained_16Cls_Acc (%)'], width, label='Chained 16-Cls Accuracy (%)', color='#1f77b4')
axes[0].bar(x_pos + width/2, df_all['Chained_F1_Macro (%)'], width, label='Chained F1 Macro (%)', color='#ff7f0e')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(df_all['Model_Name'], rotation=30, ha='right', fontsize=10)
axes[0].set_ylim(min(df_all['Chained_F1_Macro (%)'].min() - 5, 80), 101)
axes[0].set_ylabel('Persentase (%)')
axes[0].set_title('Perbandingan Akurasi & F1-Score Macro Seluruh Backbone (CASCADE)')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# 2. Scatter Plot: GFLOPs vs F1 Macro (Ukuran = Total Params Ensemble, Warna = Latensi ms)
scatter = axes[1].scatter(
    df_all['2Stage_GFLOPs'],
    df_all['Chained_F1_Macro (%)'],
    s=df_all['Cascade_Total_Params_M'] * 2.5,
    c=df_all['2Stage_Latency_ms'],
    cmap='viridis',
    alpha=0.85,
    edgecolors='black'
)
for _, row in df_all.iterrows():
    axes[1].annotate(
        row['Model_Name'],
        (row['2Stage_GFLOPs'], row['Chained_F1_Macro (%)']),
        textcoords='offset points',
        xytext=(5, 5),
        fontsize=9,
        fontweight='bold'
    )
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('2-Stage Latency (ms)')
axes[1].set_xlabel('2-Stage GFLOPs')
axes[1].set_ylabel('Chained F1 Macro (%)')
axes[1].set_title('Efisiensi Komputasi (2-Stage GFLOPs) vs Kinerja (F1 Macro)\n*Ukuran titik sebanding dengan Parameter Ensemble (M)')
axes[1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plot_out_path = os.path.join(REPORTS_BASE_DIR, 'CASCADE_All_Backbones_Visual_Comparison.png')
plt.savefig(plot_out_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'[SAVED] Grafik perbandingan komparatif disimpan di: {plot_out_path}')


In [ ]:
# --- BUNDLING HASIL MODEL (.PTH) DAN LAPORAN KE ZIP ---
bundle_target = os.path.join(WORK_DIR, 'Hasil_CASCADE_Backbones_Benchmark_Complete')
shutil.make_archive(bundle_target, 'zip', root_dir=WORK_DIR, base_dir='reports')
print(f'\n[SUCCESS] Seluruh laporan, CSV, dan grafik berhasil dikompres ke: {bundle_target}.zip')
